In [1]:
import os
import re
import gc
import json
import time
import math
import random
import warnings
import unicodedata
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

import pandas as pd
import torch
import evaluate

from tqdm.auto import tqdm
from sentence_transformers import CrossEncoder
from ollama import Client as OllamaClient

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever, TFIDFRetriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import LocalFileStore

warnings.filterwarnings("ignore")

In [2]:
@dataclass(frozen=True)
class ExperimentConfig:
    index_root: str = "outputs_index"
    vectorstore_dir: str = "faiss_index"
    docstore_dir: str = "parent_docstore"
    index_config_name: str = "index_config.json"
    documents_registry_name: str = "documents_registry.json"

    questions_path: str = "questions.json"
    results_dir: str = "outputs_experiment"

    model_name: str = "gemma3:12b"
    # model_name: str = "gemma3:27b"
    ollama_host: str = os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434")

    rounds: int = 10
    base_seed: int = 42

    retriever_k: int = 10
    top_k_context: int = 4
    top_k_after_rerank: int = 4
    cross_encoder_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"

    temperature: float = 0.1
    top_p: float = 0.9
    num_ctx: int = 8192

    bertscore_model: str = "distilbert-base-multilingual-cased"
    bertscore_lang: str = "pt"

    verbose_logs: bool = True
    show_answer_preview: bool = True
    answer_preview_chars: int = 120
    save_partial_every_round: bool = True

    @property
    def device(self) -> str:
        return "cuda" if torch.cuda.is_available() else "cpu"

CFG = ExperimentConfig()

INDEX_ROOT = Path(CFG.index_root)
VECTORSTORE_PATH = INDEX_ROOT / CFG.vectorstore_dir
DOCSTORE_PATH = INDEX_ROOT / CFG.docstore_dir
INDEX_CONFIG_PATH = INDEX_ROOT / CFG.index_config_name
DOCUMENTS_REGISTRY_PATH = INDEX_ROOT / CFG.documents_registry_name
QUESTIONS_PATH = Path(CFG.questions_path)
RESULTS_DIR = Path(CFG.results_dir)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", CFG.device)
print("Ollama host:", CFG.ollama_host)
print("Index root:", INDEX_ROOT.resolve())
print("Vectorstore:", VECTORSTORE_PATH.resolve())
print("Docstore:", DOCSTORE_PATH.resolve())
print("Questions:", QUESTIONS_PATH.resolve())
print("Results dir:", RESULTS_DIR.resolve())

Device: cuda
Ollama host: http://127.0.0.1:11434
Index root: /home/avelar/outputs_index
Vectorstore: /home/avelar/outputs_index/faiss_index
Docstore: /home/avelar/outputs_index/parent_docstore
Questions: /home/avelar/questions.json
Results dir: /home/avelar/outputs_experiment


In [3]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [4]:
def normalize_spaces(text: str) -> str:
    text = text or ""
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [5]:
def normalize_for_comparison(text: str) -> str:
    text = normalize_spaces(text).lower()
    return text

In [6]:
def token_list(text: str) -> list[str]:
    text = normalize_for_comparison(text)
    return re.findall(r"\w+", text, flags=re.UNICODE)

In [7]:
def clean_answer_text(answer: str) -> str:
    if answer is None:
        return ""
    answer = str(answer)

    junk_tokens = [
        "</s>", "<|im_start|>", "<|im_end|>", "<|eot_id|>", "<|end|>",
        "<start_of_turn>", "<end_of_turn>", "[INST]", "[/INST]",
        "assistant", "model"
    ]
    for tok in junk_tokens:
        answer = answer.replace(tok, "")

    return normalize_spaces(answer)

In [8]:
def calculate_exact_match(prediction: str, reference: str) -> float:
    return float(normalize_for_comparison(prediction) == normalize_for_comparison(reference))

In [9]:
def calculate_f1_token(prediction: str, reference: str) -> float:
    pred_tokens = set(token_list(prediction))
    ref_tokens = set(token_list(reference))

    if not pred_tokens or not ref_tokens:
        return 0.0

    common = pred_tokens.intersection(ref_tokens)
    if not common:
        return 0.0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(ref_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)

In [10]:
def normalize_source_name(text: str) -> str:
    text = str(text or "").strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace(".pdf", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [11]:
def get_doc_source(doc) -> str:
    return doc.metadata.get("source") or doc.metadata.get("file_name") or ""

In [12]:
def get_first_relevant_rank(docs, target_source: str):
    target_norm = normalize_source_name(target_source)
    for i, doc in enumerate(docs, start=1):
        source_norm = normalize_source_name(get_doc_source(doc))
        if source_norm == target_norm:
            return i
    return None

In [13]:
def hit_at_k(docs, target_source: str, k: int) -> float:
    rank = get_first_relevant_rank(docs[:k], target_source)
    return float(rank is not None)

In [14]:
def recall_at_k_single_target(docs, target_source: str, k: int) -> float:
    return hit_at_k(docs, target_source, k)

In [15]:
def reciprocal_rank(docs, target_source: str) -> float:
    rank = get_first_relevant_rank(docs, target_source)
    return 0.0 if rank is None else 1.0 / rank

In [16]:
def docs_to_sources(docs):
    return [get_doc_source(doc) for doc in docs if doc is not None]

In [17]:
def clear_memory(*objs: Any) -> None:
    for obj in objs:
        try:
            del obj
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [18]:
def save_json(obj: Any, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

In [19]:
def save_text(text: str, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

In [20]:
def load_index_config(index_config_path: Path) -> dict:
    if index_config_path.exists():
        with open(index_config_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        print("Configuração do índice carregada com sucesso.")
        return cfg

    print("⚠️ index_config.json não encontrado. Usando defaults compatíveis com seu build_index.")
    return {
        "embedding_model": "BAAI/bge-m3",
        "parent_chunk_size": 1500,
        "parent_chunk_overlap": 200,
        "child_chunk_size": 400,
        "child_chunk_overlap": 50,
        "normalize_embeddings": True,
    }

In [21]:
def build_embeddings_from_index(index_cfg: dict, device: str):
    model_name = index_cfg.get("embedding_model", "BAAI/bge-m3")
    normalize_embeddings = index_cfg.get("normalize_embeddings", True)

    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": normalize_embeddings},
    )
    return embeddings

In [22]:
def build_splitters_from_index(index_cfg: dict):
    parent_splitter = RecursiveCharacterTextSplitter(
        chunk_size=index_cfg.get("parent_chunk_size", 1500),
        chunk_overlap=index_cfg.get("parent_chunk_overlap", 200),
        add_start_index=True,
    )

    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=index_cfg.get("child_chunk_size", 400),
        chunk_overlap=index_cfg.get("child_chunk_overlap", 50),
        add_start_index=True,
    )

    return parent_splitter, child_splitter

In [23]:
def load_dense_retriever(cfg: ExperimentConfig):
    if not VECTORSTORE_PATH.exists():
        raise FileNotFoundError(f"Vectorstore não encontrado: {VECTORSTORE_PATH.resolve()}")
    if not DOCSTORE_PATH.exists():
        raise FileNotFoundError(f"Docstore não encontrado: {DOCSTORE_PATH.resolve()}")

    index_cfg = load_index_config(INDEX_CONFIG_PATH)
    embeddings = build_embeddings_from_index(index_cfg, cfg.device)
    parent_splitter, child_splitter = build_splitters_from_index(index_cfg)

    vectorstore = FAISS.load_local(
        str(VECTORSTORE_PATH),
        embeddings,
        allow_dangerous_deserialization=True
    )

    byte_store = LocalFileStore(str(DOCSTORE_PATH))

    dense_retriever = ParentDocumentRetriever(
        vectorstore=vectorstore,
        byte_store=byte_store,
        parent_splitter=parent_splitter,
        child_splitter=child_splitter,
    )
    dense_retriever.search_kwargs = {"k": cfg.retriever_k}

    return {
        "index_cfg": index_cfg,
        "embeddings": embeddings,
        "vectorstore": vectorstore,
        "byte_store": byte_store,
        "dense_retriever": dense_retriever,
    }

In [24]:
def load_parent_docs_from_dense_retriever(dense_retriever: ParentDocumentRetriever) -> list[Document]:
    if not hasattr(dense_retriever, "docstore"):
        raise RuntimeError("O retriever não expôs docstore; não foi possível montar BM25/TF-IDF.")

    docstore = dense_retriever.docstore

    if not hasattr(docstore, "yield_keys") or not hasattr(docstore, "mget"):
        raise RuntimeError("O docstore carregado não suporta yield_keys/mget.")

    keys = list(docstore.yield_keys())
    parent_docs = [doc for doc in docstore.mget(keys) if doc is not None]

    print(f"Chunks-pai carregados do docstore: {len(parent_docs)}")
    return parent_docs

In [25]:
def build_sparse_retrievers(parent_docs: list[Document], top_k: int):
    bm25_retriever = BM25Retriever.from_documents(parent_docs)
    bm25_retriever.k = top_k

    tfidf_retriever = TFIDFRetriever.from_documents(parent_docs)
    tfidf_retriever.k = top_k

    return bm25_retriever, tfidf_retriever

In [26]:
SYSTEM_PROMPT = """
Você é um assistente especialista em pergunta e resposta sobre documentos institucionais e acadêmicos.

Sua tarefa é responder perguntas usando SOMENTE o contexto recuperado.
Você deve ser extremamente fiel ao contexto.

Regras obrigatórias:
1. Use apenas informações explicitamente presentes no contexto.
2. Não invente datas, nomes, números, locais, cargas horárias, objetivos ou justificativas.
3. Se o contexto não trouxer informação suficiente, responda exatamente:
   "Não encontrei essa informação no contexto fornecido."
4. Responda em português do Brasil.
5. Responda de forma clara, natural e factual, preferencialmente em uma frase completa.
6. Quando a pergunta pedir data, ano, número, nome ou carga horária, responda em frase completa, mencionando o sujeito principal da pergunta, e não apenas o valor isolado.
7. Não mencione que recebeu instruções, não fale em "contexto acima", "trecho", "documento recuperado" ou "de acordo com o texto".
8. Não use listas, marcadores ou explicações extras, salvo se a própria pergunta exigir enumeração.
9. Evite respostas telegráficas como apenas um número, nome ou data soltos.
10. Sempre que possível, mantenha a redação próxima ao estilo de uma resposta esperada de avaliação objetiva, mas sem copiar literalmente.

Exemplos de estilo desejado:
- Pergunta: "Em que ano o curso foi criado?"
  Resposta adequada: "O curso foi criado em 2013."
- Pergunta: "Qual é a carga horária do curso?"
  Resposta adequada: "A carga horária do curso é de 3.000 horas."
- Pergunta: "Qual campus oferta o curso?"
  Resposta adequada: "O curso é ofertado no Campus de Picos."
""".strip()

In [27]:
def build_user_prompt(question: str, context: str) -> str:
    return f"""
Pergunta do usuário:
{question}

Contexto recuperado:
{context}

Responda à pergunta com base somente no contexto recuperado.

Importante:
- escreva a resposta em uma frase completa;
- mencione explicitamente o assunto principal da pergunta;
- não responda apenas com um número, nome ou data isolados;
- seja natural, objetiva e fiel ao contexto.
""".strip()

In [28]:
def build_ollama_client(cfg: ExperimentConfig) -> OllamaClient:
    client = OllamaClient(host=cfg.ollama_host)
    return client

In [29]:
def check_ollama_model(client: OllamaClient, model_name: str) -> None:
    try:
        _ = client.show(model_name)
        print(f"Modelo disponível no Ollama: {model_name}")
    except Exception as e:
        raise RuntimeError(
            f"Não consegui acessar o modelo '{model_name}' em {CFG.ollama_host}. "
            f"Verifique se o Ollama está de pé e se o modelo está puxado no servidor.\nErro: {e}"
        )

In [30]:
def generate_answer_with_ollama(
    client: OllamaClient,
    cfg: ExperimentConfig,
    question: str,
    context: str,
    seed: int,
) -> str:
    response = client.chat(
        model=cfg.model_name,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(question, context)},
        ],
        options={
            "temperature": cfg.temperature,
            "top_p": cfg.top_p,
            "seed": seed,
            "num_ctx": cfg.num_ctx,
        },
    )
    answer = response["message"]["content"]
    return clean_answer_text(answer)

In [31]:
def format_doc_for_context(doc: Document, idx: int) -> str:
    source = doc.metadata.get("source") or doc.metadata.get("file_name") or "desconhecido"
    content = normalize_spaces(doc.page_content)
    return f"[Trecho {idx} | Fonte: {source}]\n{content}"

In [32]:
def build_context_from_docs(docs: list[Document], top_k: int) -> tuple[str, list[Document]]:
    docs = [doc for doc in docs if doc is not None][:top_k]
    context_parts = [format_doc_for_context(doc, i + 1) for i, doc in enumerate(docs)]
    context = "\n\n".join(context_parts).strip()
    return context, docs

In [33]:
def extract_sources(docs: list[Document]) -> list[str]:
    sources = []
    for doc in docs:
        source = doc.metadata.get("source") or doc.metadata.get("file_name")
        if source:
            sources.append(source)
    return list(dict.fromkeys(sources))

In [34]:
def rerank_documents(
    question: str,
    retrieved_docs: list[Document],
    cross_encoder: CrossEncoder,
    top_k: int,
) -> list[Document]:
    if not retrieved_docs:
        return []

    pairs = [[question, normalize_spaces(doc.page_content)] for doc in retrieved_docs]
    scores = cross_encoder.predict(pairs)

    doc_scores = list(zip(retrieved_docs, scores))
    doc_scores = sorted(doc_scores, key=lambda x: x[1], reverse=True)

    reranked_docs = [doc for doc, _ in doc_scores[:top_k]]
    return reranked_docs

In [35]:
def ask_dense_no_rerank(
    question: str,
    dense_retriever: ParentDocumentRetriever,
) -> list[Document]:
    docs = dense_retriever.invoke(question)
    return docs

In [36]:
def ask_dense_with_rerank(
    question: str,
    dense_retriever: ParentDocumentRetriever,
    cross_encoder: CrossEncoder,
) -> tuple[list[Document], list[Document]]:
    initial_docs = dense_retriever.invoke(question)

    reranked_docs = rerank_documents(
        question=question,
        retrieved_docs=initial_docs,
        cross_encoder=cross_encoder,
        top_k=len(initial_docs),
    )

    return initial_docs, reranked_docs

In [37]:
def ask_sparse(
    question: str,
    sparse_retriever,
) -> list[Document]:
    docs = sparse_retriever.invoke(question)
    return docs

In [38]:
def run_single_qa(
    client: OllamaClient,
    cfg: ExperimentConfig,
    method_name: str,
    question_item: dict,
    round_id: int,
    round_seed: int,
    dense_retriever: ParentDocumentRetriever,
    cross_encoder: CrossEncoder,
    bm25_retriever,
    tfidf_retriever,
) -> dict:
    question = question_item["pergunta"]
    expected_answer = question_item["resposta_esperada"]
    target_document = question_item.get("documento", "")

    start_time = time.perf_counter()

    if method_name == "dense_no_rerank":
        initial_docs = ask_dense_no_rerank(question, dense_retriever)
        ranked_docs = initial_docs

    elif method_name == "dense_rerank":
        initial_docs, ranked_docs = ask_dense_with_rerank(
            question=question,
            dense_retriever=dense_retriever,
            cross_encoder=cross_encoder,
        )

    elif method_name == "bm25":
        initial_docs = ask_sparse(question, bm25_retriever)
        ranked_docs = initial_docs

    elif method_name == "tfidf":
        initial_docs = ask_sparse(question, tfidf_retriever)
        ranked_docs = initial_docs

    else:
        raise ValueError(f"Método desconhecido: {method_name}")

    selected_docs = [doc for doc in ranked_docs if doc is not None][:cfg.top_k_context]
    context, selected_docs = build_context_from_docs(selected_docs, cfg.top_k_context)

    initial_sources = docs_to_sources(initial_docs)
    ranked_sources = docs_to_sources(ranked_docs)
    final_context_sources = docs_to_sources(selected_docs)

    rank_initial = get_first_relevant_rank(initial_docs, target_document)
    rank_final = get_first_relevant_rank(ranked_docs, target_document)

    answer = generate_answer_with_ollama(
        client=client,
        cfg=cfg,
        question=question,
        context=context,
        seed=round_seed,
    )

    latency_sec = time.perf_counter() - start_time

    return {
        "rodada": round_id,
        "seed": round_seed,
        "metodo": method_name,
        "documento": target_document,
        "pergunta": question,
        "resposta_esperada": expected_answer,
        "resposta_gerada": answer,

        "contexto_recuperado": context,
        "fontes_recuperadas": json.dumps(final_context_sources, ensure_ascii=False),
        "ranked_sources_initial": json.dumps(initial_sources, ensure_ascii=False),
        "ranked_sources_final": json.dumps(ranked_sources, ensure_ascii=False),

        "rank_initial": rank_initial,
        "rank_final": rank_final,

        "rr_initial": 0.0 if rank_initial is None else 1.0 / rank_initial,
        "rr_final": 0.0 if rank_final is None else 1.0 / rank_final,

        "hit@1": hit_at_k(ranked_docs, target_document, 1),
        "hit@3": hit_at_k(ranked_docs, target_document, 3),
        "hit@5": hit_at_k(ranked_docs, target_document, 5),
        "hit@10": hit_at_k(ranked_docs, target_document, 10),

        "recall@1": recall_at_k_single_target(ranked_docs, target_document, 1),
        "recall@3": recall_at_k_single_target(ranked_docs, target_document, 3),
        "recall@5": recall_at_k_single_target(ranked_docs, target_document, 5),
        "recall@10": recall_at_k_single_target(ranked_docs, target_document, 10),

        "source_in_final_context": hit_at_k(selected_docs, target_document, len(selected_docs)),

        "n_retrieved_initial": len(initial_docs),
        "n_retrieved_final": len(ranked_docs),
        "n_fontes": len(final_context_sources),
        "latencia_segundos": round(latency_sec, 4),
    }

In [39]:
for var_name in [
    "cross_encoder",
    "dense_retriever",
    "parent_docs",
    "bm25_retriever",
    "tfidf_retriever",
    "resources",
]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Limpeza concluída.")

Limpeza concluída.


In [40]:
def evaluate_methods_by_question(df_results: pd.DataFrame, cfg: ExperimentConfig):
    rouge_metric = evaluate.load("rouge")
    bleu_metric = evaluate.load("bleu")
    bertscore_metric = evaluate.load("bertscore")

    metric_rows = []

    for _, row in tqdm(df_results.iterrows(), total=len(df_results), desc="Métricas por questão"):
        prediction = str(row["resposta_gerada"] if pd.notna(row["resposta_gerada"]) else "")
        reference = str(row["resposta_esperada"] if pd.notna(row["resposta_esperada"]) else "")

        rouge_scores = rouge_metric.compute(
            predictions=[prediction],
            references=[reference]
        )

        bleu_scores = bleu_metric.compute(
            predictions=[prediction],
            references=[[reference]]
        )

        bertscore_scores = bertscore_metric.compute(
            predictions=[prediction],
            references=[reference],
            lang=cfg.bertscore_lang,
            model_type=cfg.bertscore_model,
            device="cpu",
            batch_size=1,
        )

        f1_score = calculate_f1_token(prediction, reference)
        em_score = calculate_exact_match(prediction, reference)

        metric_rows.append({
            "rodada": row["rodada"],
            "seed": row["seed"],
            "metodo": row["metodo"],
            "documento": row["documento"],
            "pergunta": row["pergunta"],
            "resposta_esperada": reference,
            "resposta_gerada": prediction,
            "rouge1": rouge_scores["rouge1"],
            "rouge2": rouge_scores["rouge2"],
            "rougeL": rouge_scores["rougeL"],
            "rougeLsum": rouge_scores.get("rougeLsum", float("nan")),
            "bleu": bleu_scores["bleu"],
            "bertscore_precision": float(bertscore_scores["precision"][0]),
            "bertscore_recall": float(bertscore_scores["recall"][0]),
            "bertscore_f1": float(bertscore_scores["f1"][0]),
            "f1_token": float(f1_score),
            "exact_match": float(em_score),
            "latencia_segundos": row["latencia_segundos"],
            "n_fontes": row["n_fontes"],
        })

    df_question_metrics = pd.DataFrame(metric_rows)

    question_agg_cols = [
        "rouge1", "rouge2", "rougeL", "rougeLsum",
        "bleu",
        "bertscore_precision", "bertscore_recall", "bertscore_f1",
        "f1_token", "exact_match",
        "latencia_segundos", "n_fontes",
    ]

    df_question_summary = (
        df_question_metrics
        .groupby(["metodo", "documento", "pergunta"], dropna=False)[question_agg_cols]
        .agg(["mean", "std"])
        .reset_index()
    )

    df_question_summary.columns = [
        f"{col}_{stat}" if stat else col
        for col, stat in df_question_summary.columns
    ]

    df_method_summary = (
        df_question_metrics
        .groupby("metodo", dropna=False)[question_agg_cols]
        .agg(["mean", "std"])
        .reset_index()
    )

    df_method_summary.columns = [
        f"{col}_{stat}" if stat else col
        for col, stat in df_method_summary.columns
    ]

    return df_question_metrics, df_question_summary, df_method_summary

In [41]:
seed_everything(CFG.base_seed)

print("Carregando stack do índice...")
resources = load_dense_retriever(CFG)

dense_retriever = resources["dense_retriever"]
parent_docs = load_parent_docs_from_dense_retriever(dense_retriever)
bm25_retriever, tfidf_retriever = build_sparse_retrievers(parent_docs, CFG.retriever_k)

print("Carregando cross-encoder...")
cross_encoder = CrossEncoder(
    CFG.cross_encoder_model,
    device=CFG.device
)

print("Conectando ao Ollama...")
ollama_client = build_ollama_client(CFG)
check_ollama_model(ollama_client, CFG.model_name)

print("Carregando perguntas...")
with open(QUESTIONS_PATH, "r", encoding="utf-8") as f:
    evaluation_dataset = json.load(f)

required_fields = {"documento", "pergunta", "resposta_esperada"}
for i, item in enumerate(evaluation_dataset):
    missing = required_fields - set(item.keys())
    if missing:
        raise ValueError(f"Pergunta na posição {i} está sem os campos obrigatórios: {missing}")

print(f"Perguntas carregadas: {len(evaluation_dataset)}")

test_item = evaluation_dataset[0]
test_result = run_single_qa(
    client=ollama_client,
    cfg=CFG,
    method_name="dense_rerank",
    question_item=test_item,
    round_id=1,
    round_seed=CFG.base_seed + 1,
    dense_retriever=dense_retriever,
    cross_encoder=cross_encoder,
    bm25_retriever=bm25_retriever,
    tfidf_retriever=tfidf_retriever,
)

Carregando stack do índice...
Configuração do índice carregada com sucesso.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Chunks-pai carregados do docstore: 12425
Carregando cross-encoder...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Conectando ao Ollama...
Modelo disponível no Ollama: gemma3:12b
Carregando perguntas...
Perguntas carregadas: 20


In [42]:
print("\nPERGUNTA:")
print(test_result["pergunta"])


PERGUNTA:
Em que ano o Curso Superior de Tecnologia em Análise e Desenvolvimento de Sistemas foi criado no Campus de Picos?


In [43]:
print("\nRESPOSTA ESPERADA:")
print(test_result["resposta_esperada"])


RESPOSTA ESPERADA:
O Curso Superior de Tecnologia em Análise e Desenvolvimento de Sistemas foi criado no Campus de Picos em 2013.


In [44]:
print("\nRESPOSTA GERADA:")
print(test_result["resposta_gerada"])


RESPOSTA GERADA:
O Curso Superior de Tecnologia em Análise e Desenvolvimento de Sistemas foi criado no Campus de Picos em 2013.


In [45]:
print("\nFONTES:")
print(test_result["fontes_recuperadas"])


FONTES:
["CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DESENVOLVIMENTO_DE_SISTEMAS.pdf", "RESOLUÇÃO Nº 59-2025- Aprova a Criação do Curso de Licenciatura em  Pedagogia,  no âmbito do Instituto Federal de Educação, Ciência e Tecnologia do Piauí (IFPI) e Anexo.pdf", "RESOLUÇÃO Nº 37-2024-Aprova a criação do curso de Especialização em Gestão Escolar, modalidade de Educação a Distância (EaD), no âmbito do IFPI e Anexo.pdf", "RESOLUÇÃO NORMATIVA Nº 196-2024 - Atualiza a Estrutura Organizacional do Instituto Federal de Educação, Ciência e Tecnologia do Piauí (IFPI) e Anexo -  REVOGADA PELA RESOLUÇÃO NORMATIVA Nº 224_2024 .pdf"]


In [46]:
print("\nPRIMEIROS 1200 CARACTERES DO CONTEXTO:")
print(test_result["contexto_recuperado"][:1200])


PRIMEIROS 1200 CARACTERES DO CONTEXTO:
[Trecho 1 | Fonte: CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DESENVOLVIMENTO_DE_SISTEMAS.pdf]
do ano referido, com uma oferta de 64 vagas, distribuídas igualmente entres os turnos tarde e noite. Os cursos de Tecnologia da Informação foram aperfeiçoados e receberam outra denominação: Análise e Desenvolvimento de Sistemas. Em 2002 foi autorizado a criação do Curso Superior de Tecnologia em Análise e Desenvolvimento de Sistemas no Campus Teresina Central, em 2006 no Campus de Floriano, em 2013 no Campus de Picos, 2018 no Campus de Corrente, 2020 no Campus de Pedro II e o mais recente em Parnaíba que iniciou em 2022. 1.7 Justificativa de Oferta do Curso e demandas sociais no mundo do trabalho A identificação das necessidades de conhecimento, habilidades e atitudes apresenta-se como uma preocupação permanente das áreas/instituições envolvidas com a oferta de produtos/serviços de sistemas e processos de gestão empresarial. Independente do seu porte, n

In [47]:
def preview_text(text: str, max_chars: int = 120) -> str:
    text = normalize_spaces(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "..."

In [48]:
def print_question_log(row: dict, question_idx: int, total_questions: int, cfg: ExperimentConfig) -> None:
    if not cfg.verbose_logs:
        return

    msg = (
        f"[Rodada {row['rodada']:02d}] "
        f"[{row['metodo']}] "
        f"[Pergunta {question_idx:02d}/{total_questions:02d}] "
        f"lat={row['latencia_segundos']:.2f}s | "
        f"fontes={row['n_fontes']} | "
        f"doc={row['documento']}"
    )
    print(msg)

    if cfg.show_answer_preview:
        print("  resposta:", preview_text(row["resposta_gerada"], cfg.answer_preview_chars))

In [49]:
def summarize_partial_rows(rows: list[dict]) -> dict:
    if not rows:
        return {
            "n": 0,
            "lat_media": float("nan"),
            "fontes_media": float("nan"),
            "exact_match": float("nan"),
            "f1_token": float("nan"),
        }

    latencias = [r["latencia_segundos"] for r in rows if pd.notna(r["latencia_segundos"])]
    fontes = [r["n_fontes"] for r in rows]

    em_scores = [
        calculate_exact_match(r["resposta_gerada"], r["resposta_esperada"])
        for r in rows
    ]
    f1_scores = [
        calculate_f1_token(r["resposta_gerada"], r["resposta_esperada"])
        for r in rows
    ]

    return {
        "n": len(rows),
        "lat_media": float(sum(latencias) / len(latencias)) if latencias else float("nan"),
        "fontes_media": float(sum(fontes) / len(fontes)) if fontes else float("nan"),
        "exact_match": float(sum(em_scores) / len(em_scores)) if em_scores else float("nan"),
        "f1_token": float(sum(f1_scores) / len(f1_scores)) if f1_scores else float("nan"),
    }

In [50]:
def print_method_summary(rodada: int, metodo: str, rows: list[dict]) -> None:
    stats = summarize_partial_rows(rows)
    print(
        f"\n>>> Resumo parcial | rodada={rodada:02d} | metodo={metodo}"
        f" | n={stats['n']}"
        f" | lat_media={stats['lat_media']:.2f}s"
        f" | fontes_media={stats['fontes_media']:.2f}"
        f" | EM={stats['exact_match']:.4f}"
        f" | F1_token={stats['f1_token']:.4f}"
    )

In [51]:
def print_round_summary(rodada: int, round_rows: list[dict]) -> None:
    df_round = pd.DataFrame(round_rows)
    if df_round.empty:
        return

    resumo = (
        df_round.groupby("metodo", dropna=False)
        .apply(
            lambda x: pd.Series({
                "n_questoes": len(x),
                "lat_media": x["latencia_segundos"].mean(),
                "fontes_media": x["n_fontes"].mean(),
                "exact_match": sum(
                    calculate_exact_match(p, r)
                    for p, r in zip(x["resposta_gerada"], x["resposta_esperada"])
                ) / len(x),
                "f1_token": sum(
                    calculate_f1_token(p, r)
                    for p, r in zip(x["resposta_gerada"], x["resposta_esperada"])
                ) / len(x),
            })
        )
        .reset_index()
    )

    print(f"\n{'='*90}")
    print(f"RESUMO DA RODADA {rodada:02d}")
    print(f"{'='*90}")
    print(resumo.round(4).to_string(index=False))
    print(f"{'='*90}\n")

In [52]:
METHODS = [
    "dense_no_rerank",
    "dense_rerank",
    "bm25",
    "tfidf",
]

all_results = []

for rodada in tqdm(range(1, CFG.rounds + 1), desc=f"Rodadas | modelo={CFG.model_name}"):
    round_seed = CFG.base_seed + rodada
    seed_everything(round_seed)

    round_rows = []

    print(f"\n{'#'*100}")
    print(f"Iniciando rodada {rodada}/{CFG.rounds} | modelo={CFG.model_name} | seed={round_seed}")
    print(f"{'#'*100}")

    for metodo in tqdm(METHODS, desc=f"Métodos da rodada {rodada}", leave=False):
        method_rows = []

        print(f"\n--- Rodada {rodada:02d} | Método: {metodo} ---")

        for question_idx, item in enumerate(
            tqdm(evaluation_dataset, desc=f"Perguntas | {metodo}", leave=False),
            start=1
        ):
            try:
                row = run_single_qa(
                    client=ollama_client,
                    cfg=CFG,
                    method_name=metodo,
                    question_item=item,
                    round_id=rodada,
                    round_seed=round_seed,
                    dense_retriever=dense_retriever,
                    cross_encoder=cross_encoder,
                    bm25_retriever=bm25_retriever,
                    tfidf_retriever=tfidf_retriever,
                )
            except Exception as e:
                row = {
                    "rodada": rodada,
                    "seed": round_seed,
                    "metodo": metodo,
                    "documento": item.get("documento", ""),
                    "pergunta": item.get("pergunta", ""),
                    "resposta_esperada": item.get("resposta_esperada", ""),
                    "resposta_gerada": f"ERRO: {e}",
                    "contexto_recuperado": "",
                    "fontes_recuperadas": "[]",
                    "n_fontes": 0,
                    "latencia_segundos": float("nan"),
                }

            all_results.append(row)
            round_rows.append(row)
            method_rows.append(row)

            print_question_log(
                row=row,
                question_idx=question_idx,
                total_questions=len(evaluation_dataset),
                cfg=CFG,
            )

        print_method_summary(rodada, metodo, method_rows)

    print_round_summary(rodada, round_rows)

    if CFG.save_partial_every_round:
        partial_csv = RESULTS_DIR / f"rag_resultados_parciais_ate_rodada_{rodada:02d}.csv"
        pd.DataFrame(all_results).to_csv(partial_csv, index=False, encoding="utf-8")
        print(f"CSV parcial salvo em: {partial_csv.resolve()}")

df_results = pd.DataFrame(all_results)

results_csv = RESULTS_DIR / "rag_resultados_brutos.csv"
df_results.to_csv(results_csv, index=False, encoding="utf-8")

print(f"Resultados brutos salvos em: {results_csv.resolve()}")
print(df_results.head())

Rodadas | modelo=gemma3:12b:   0%|          | 0/10 [00:00<?, ?it/s]


####################################################################################################
Iniciando rodada 1/10 | modelo=gemma3:12b | seed=43
####################################################################################################


Métodos da rodada 1:   0%|          | 0/4 [00:00<?, ?it/s]


--- Rodada 01 | Método: dense_no_rerank ---


Perguntas | dense_no_rerank:   0%|          | 0/20 [00:00<?, ?it/s]

...

[Rodada 10] [dense_rerank] [Pergunta 01/20] lat=3.70s | fontes=4 | doc=CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DESENVOLVIMENTO_DE_SISTEMAS.pdf
  resposta: O Curso Superior de Tecnologia em Análise e Desenvolvimento de Sistemas foi criado no Campus de Picos em 2013.
[Rodada 10] [dense_rerank] [Pergunta 02/20] lat=4.18s | fontes=4 | doc=CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DESENVOLVIMENTO_DE_SISTEMAS.pdf
  resposta: O Projeto Pedagógico do Curso Superior de Tecnologia em Análise e Desenvolvimento de Sistemas é fruto de um processo de ...
[Rodada 10] [dense_rerank] [Pergunta 03/20] lat=4.02s | fontes=4 | doc=CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DESENVOLVIMENTO_DE_SISTEMAS.pdf
  resposta: A carga horária total do Curso Superior de Tecnologia em Análise e Desenvolvimento de Sistemas é de 400 horas, distribuí...
[Rodada 10] [dense_rerank] [Pergunta 04/20] lat=3.84s | fontes=4 | doc=CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DESENVOLVIMENTO_DE_SISTEMAS.pdf
  resposta: O Curso S

Perguntas | bm25:   0%|          | 0/20 [00:00<?, ?it/s]

[Rodada 10] [bm25] [Pergunta 01/20] lat=3.59s | fontes=4 | doc=CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DESENVOLVIMENTO_DE_SISTEMAS.pdf
  resposta: O Curso Superior de Tecnologia em Análise e Desenvolvimento de Sistemas no Campus de Picos foi autorizado em 2013.
[Rodada 10] [bm25] [Pergunta 02/20] lat=4.44s | fontes=4 | doc=CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DESENVOLVIMENTO_DE_SISTEMAS.pdf
  resposta: A unidade de oferta do Curso Superior de Tecnologia em Análise e Desenvolvimento de Sistemas descrita no PPC são os camp...
[Rodada 10] [bm25] [Pergunta 03/20] lat=4.41s | fontes=4 | doc=CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DESENVOLVIMENTO_DE_SISTEMAS.pdf
  resposta: As atividades complementares do Curso Superior de Tecnologia em Análise e Desenvolvimento de Sistemas deverão ter o tota...
[Rodada 10] [bm25] [Pergunta 04/20] lat=3.43s | fontes=4 | doc=CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DESENVOLVIMENTO_DE_SISTEMAS.pdf
  resposta: O Curso Superior de Tecnologia em Aná

Perguntas | tfidf:   0%|          | 0/20 [00:00<?, ?it/s]

[Rodada 10] [tfidf] [Pergunta 01/20] lat=3.49s | fontes=4 | doc=CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DESENVOLVIMENTO_DE_SISTEMAS.pdf
  resposta: O Curso Superior de Tecnologia em Análise e Desenvolvimento de Sistemas foi criado no Campus de Picos em 2013.
[Rodada 10] [tfidf] [Pergunta 02/20] lat=4.14s | fontes=4 | doc=CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DESENVOLVIMENTO_DE_SISTEMAS.pdf
  resposta: O Curso Superior de Tecnologia em Análise e Desenvolvimento de Sistemas contribui com a demanda de profissionais para at...
[Rodada 10] [tfidf] [Pergunta 03/20] lat=2.53s | fontes=4 | doc=CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DESENVOLVIMENTO_DE_SISTEMAS.pdf
  resposta: Não encontrei essa informação no contexto fornecido.
[Rodada 10] [tfidf] [Pergunta 04/20] lat=3.67s | fontes=4 | doc=CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DESENVOLVIMENTO_DE_SISTEMAS.pdf
  resposta: O Curso Superior de Tecnologia em Análise e Desenvolvimento de Sistemas teve uma oferta de 64 vagas, distrib

In [53]:
retrieval_metric_cols = [
    "hit@1", "hit@3", "hit@5", "hit@10",
    "recall@1", "recall@3", "recall@5", "recall@10",
    "rr_final",
    "source_in_final_context",
]

df_retrieval_summary = (
    df_results
    .groupby("metodo", dropna=False)[retrieval_metric_cols]
    .agg(["mean", "std"])
    .reset_index()
)

df_retrieval_summary.columns = [
    f"{col}_{stat}" if stat else col
    for col, stat in df_retrieval_summary.columns
]

df_retrieval_summary = df_retrieval_summary.rename(
    columns={"rr_final_mean": "mrr_mean", "rr_final_std": "mrr_std"}
)

retrieval_summary_csv = RESULTS_DIR / "rag_metricas_recuperacao_por_metodo.csv"
df_retrieval_summary.to_csv(retrieval_summary_csv, index=False, encoding="utf-8")

print("\n===== MÉTRICAS DE RECUPERAÇÃO POR MÉTODO =====")
display(df_retrieval_summary.round(4))
print(f"Salvo em: {retrieval_summary_csv.resolve()}")


===== MÉTRICAS DE RECUPERAÇÃO POR MÉTODO =====


,metodo,hit@1_mean,hit@1_std,hit@3_mean,hit@3_std,hit@5_mean,hit@5_std,hit@10_mean,hit@10_std,recall@1_mean,...,recall@3_mean,recall@3_std,recall@5_mean,recall@5_std,recall@10_mean,recall@10_std,mrr_mean,mrr_std,source_in_final_context_mean,source_in_final_context_std
0,bm25,0.35,0.4782,0.55,0.4987,0.55,0.4987,0.65,0.4782,0.35,...,0.55,0.4987,0.55,0.4987,0.65,0.4782,0.4542,0.4351,0.55,0.4987
1,dense_no_rerank,0.70,0.4594,0.75,0.4341,0.80,0.4010,0.95,0.2185,0.70,...,0.75,0.4341,0.80,0.4010,0.95,0.2185,0.7597,0.3776,0.80,0.4010
2,dense_rerank,0.80,0.4010,0.80,0.4010,0.90,0.3008,0.95,0.2185,0.80,...,0.80,0.4010,0.90,0.3008,0.95,0.2185,0.8308,0.3418,0.85,0.3580
3,tfidf,0.65,0.4782,0.75,0.4341,0.75,0.4341,0.75,0.4341,0.65,...,0.75,0.4341,0.75,0.4341,0.75,0.4341,0.7000,0.4312,0.75,0.4341


Salvo em: /home/avelar/outputs_experiment/rag_metricas_recuperacao_por_metodo.csv


In [54]:
df_metricas_questao, df_resumo_questao, df_resumo_metodo = evaluate_methods_by_question(df_results, CFG)

metrics_question_csv = RESULTS_DIR / "rag_metricas_por_questao.csv"
metrics_question_summary_csv = RESULTS_DIR / "rag_metricas_resumo_por_questao.csv"
metrics_method_summary_csv = RESULTS_DIR / "rag_metricas_resumo_por_metodo.csv"
run_config_json = RESULTS_DIR / "rag_run_config.json"

df_metricas_questao.to_csv(metrics_question_csv, index=False, encoding="utf-8")
df_resumo_questao.to_csv(metrics_question_summary_csv, index=False, encoding="utf-8")
df_resumo_metodo.to_csv(metrics_method_summary_csv, index=False, encoding="utf-8")
save_json(asdict(CFG), run_config_json)

print("Arquivos salvos:")
print(metrics_question_csv.resolve())
print(metrics_question_summary_csv.resolve())
print(metrics_method_summary_csv.resolve())
print(run_config_json.resolve())

print("\n===== MÉTRICAS POR QUESTÃO =====")
display(df_metricas_questao.round(4))

print("\n===== RESUMO POR QUESTÃO (MÉDIA E DESVIO-PADRÃO NAS RODADAS) =====")
display(df_resumo_questao.round(4))

print("\n===== RESUMO FINAL POR MÉTODO =====")
display(df_resumo_metodo.round(4))

Métricas por questão:   0%|          | 0/800 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Arquivos salvos:
/home/avelar/outputs_experiment/rag_metricas_por_questao.csv
/home/avelar/outputs_experiment/rag_metricas_resumo_por_questao.csv
/home/avelar/outputs_experiment/rag_metricas_resumo_por_metodo.csv
/home/avelar/outputs_experiment/rag_run_config.json

===== MÉTRICAS POR QUESTÃO =====


,rodada,seed,metodo,documento,pergunta,resposta_esperada,resposta_gerada,rouge1,rouge2,rougeL,rougeLsum,bleu,bertscore_precision,bertscore_recall,bertscore_f1,f1_token,exact_match,latencia_segundos,n_fontes
0,1,43,dense_no_rerank,CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DES...,Em que ano o Curso Superior de Tecnologia em A...,O Curso Superior de Tecnologia em Análise e De...,O Curso Superior de Tecnologia em Análise e De...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0,3.7981,4
1,1,43,dense_no_rerank,CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DES...,Qual é a unidade de oferta do Curso Superior d...,A unidade de oferta do curso é o Campus Teresi...,A unidade da oferta do Curso Superior de Tecno...,0.6667,0.4286,0.6000,0.6000,0.2832,0.8715,0.9716,0.9188,0.7333,0.0,3.4666,4
2,1,43,dense_no_rerank,CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DES...,Qual é a carga horária total do Curso Superior...,A carga horária total do curso é de 2100 horas.,A carga horária total do Curso Superior de Tec...,0.6429,0.4615,0.6429,0.6429,0.2577,0.8713,0.9635,0.9151,0.6667,0.0,3.5462,4
3,1,43,dense_no_rerank,CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DES...,Quantas vagas por ano são ofertadas no Curso S...,O curso oferta 40 vagas por ano.,O Curso Superior de Tecnologia em Análise e De...,0.4444,0.3200,0.4444,0.4444,0.1326,0.8292,0.9554,0.8878,0.4800,0.0,3.5638,4
4,1,43,dense_no_rerank,RESOLUÇÃO Nº 14-2025-Aprova a Criação do Curso...,Qual é a carga horária total do curso Escola d...,A carga horária total do curso Escola da Terra...,A carga horária total do curso Escola da Terra...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0,3.5108,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,10,52,tfidf,"RESOLUÇÃO NORMATIVA Nº 218-2024-Estab. normas,...",Qual é o limite de carga horária para atividad...,As atividades remuneradas eventuais não podem ...,Não encontrei essa informação no contexto forn...,0.1667,0.0909,0.1667,0.1667,0.0000,0.8234,0.8058,0.8145,0.1000,0.0,2.9279,4
796,10,52,tfidf,RESOLUÇÃO NORMATIVA Nº 217-2024-Atualiza o Reg...,Como é composta a CPPD do IFPI em número de me...,"A CPPD do IFPI é composta por 10 docentes, sen...",A CPPD do IFPI terá à sua disposição quatro do...,0.4348,0.1818,0.3913,0.3913,0.1086,0.8727,0.9457,0.9077,0.4762,0.0,4.3961,4
797,10,52,tfidf,RESOLUÇÃO NORMATIVA Nº 217-2024-Atualiza o Reg...,Qual é o mandato dos integrantes da CPPD do IFPI?,"O mandato dos integrantes da CPPD é de 4 anos,...",O mandato dos integrantes da CPPD do IFPI não ...,0.4118,0.3125,0.4118,0.4118,0.2332,0.8847,0.8674,0.8760,0.4375,0.0,3.1908,4
798,10,52,tfidf,RESOLUÇÃO Nº 27-2024-Aprova a criação do curs...,Quantas vagas são ofertadas no curso FIC em Aq...,O curso FIC em Aquicultor oferta 40 vagas.,O curso de Qualificação Profissional - Formaçã...,0.5600,0.3478,0.5600,0.5600,0.0000,0.8636,0.9696,0.9135,0.6087,0.0,3.8712,4



===== RESUMO POR QUESTÃO (MÉDIA E DESVIO-PADRÃO NAS RODADAS) =====


,metodo,documento,pergunta,rouge1_mean,rouge1_std,rouge2_mean,rouge2_std,rougeL_mean,rougeL_std,rougeLsum_mean,...,bertscore_f1_mean,bertscore_f1_std,f1_token_mean,f1_token_std,exact_match_mean,exact_match_std,latencia_segundos_mean,latencia_segundos_std,n_fontes_mean,n_fontes_std
0,bm25,CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DES...,Em que ano o Curso Superior de Tecnologia em A...,0.9600,0.0211,0.8316,0.0888,0.9200,0.0422,0.9200,...,0.9912,0.0047,0.9500,0.0264,0.2,0.4216,3.5793,0.0097,4.0,0.0
1,bm25,CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DES...,Qual é a carga horária total do Curso Superior...,0.3913,0.0000,0.2273,0.0000,0.3043,0.0000,0.3043,...,0.8681,0.0000,0.4324,0.0000,0.0,0.0000,4.4139,0.0079,4.0,0.0
2,bm25,CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DES...,Qual é a unidade de oferta do Curso Superior d...,0.4172,0.0029,0.2966,0.0084,0.4172,0.0029,0.4172,...,0.8841,0.0002,0.4368,0.0333,0.0,0.0000,4.2475,0.3656,4.0,0.0
3,bm25,CURSO_SUPERIOR_DE_TECNOLOGIA_EM_ANÁLISE_E_DES...,Quantas vagas por ano são ofertadas no Curso S...,0.4511,0.0107,0.3097,0.0166,0.4511,0.0107,0.4511,...,0.8902,0.0038,0.4916,0.0186,0.0,0.0000,3.4789,0.0807,4.0,0.0
4,bm25,RESOLUÇÃO NORMATIVA Nº 217-2024-Atualiza o Reg...,Como é composta a CPPD do IFPI em número de me...,0.5625,0.0000,0.4000,0.0000,0.5625,0.0000,0.5625,...,0.9510,0.0000,0.6452,0.0000,0.0,0.0000,3.4694,0.0059,4.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,tfidf,RESOLUÇÃO Nº 14-2025-Aprova a Criação do Curso...,Qual é a meta física do curso Escola da Terra ...,0.3721,0.0000,0.1463,0.0000,0.2326,0.0000,0.2326,...,0.8764,0.0000,0.4000,0.0000,0.0,0.0000,4.5311,0.0122,4.0,0.0
76,tfidf,RESOLUÇÃO Nº 14-2025-Aprova a Criação do Curso...,Quando está previsto o início do curso Escola ...,0.5806,0.0000,0.3448,0.0000,0.5806,0.0000,0.5806,...,0.9134,0.0000,0.6429,0.0000,0.0,0.0000,3.4187,0.0050,4.0,0.0
77,tfidf,RESOLUÇÃO Nº 14-2025-Aprova a Criação do Curso...,Quem é o coordenador do curso Escola da Terra?,0.1000,0.0000,0.0000,0.0000,0.1000,0.0000,0.1000,...,0.8339,0.0000,0.0000,0.0000,0.0,0.0000,2.8850,0.0091,4.0,0.0
78,tfidf,RESOLUÇÃO Nº 27-2024-Aprova a criação do curs...,Qual é a forma de acesso ao curso FIC em Aquic...,0.4205,0.0060,0.2556,0.0039,0.3004,0.0043,0.3004,...,0.8929,0.0001,0.4955,0.0073,0.0,0.0000,4.0430,0.0297,4.0,0.0



===== RESUMO FINAL POR MÉTODO =====


,metodo,rouge1_mean,rouge1_std,rouge2_mean,rouge2_std,rougeL_mean,rougeL_std,rougeLsum_mean,rougeLsum_std,bleu_mean,...,bertscore_f1_mean,bertscore_f1_std,f1_token_mean,f1_token_std,exact_match_mean,exact_match_std,latencia_segundos_mean,latencia_segundos_std,n_fontes_mean,n_fontes_std
0,bm25,0.4973,0.2495,0.3421,0.2803,0.4598,0.2615,0.4598,0.2615,0.2249,...,0.8994,0.0493,0.5168,0.2773,0.06,0.2381,3.7916,0.6490,4.0,0.0
1,dense_no_rerank,0.6480,0.2328,0.5314,0.2897,0.6300,0.2394,0.6300,0.2394,0.3901,...,0.9257,0.0457,0.6794,0.2177,0.20,0.4010,4.0419,0.5831,4.0,0.0
2,dense_rerank,0.5705,0.2393,0.4414,0.2987,0.5481,0.2515,0.5481,0.2515,0.3110,...,0.9139,0.0476,0.6156,0.2252,0.10,0.3008,4.3653,0.8884,4.0,0.0
3,tfidf,0.4251,0.2414,0.2865,0.2565,0.3998,0.2499,0.3998,0.2499,0.1557,...,0.8867,0.0434,0.4561,0.2551,0.05,0.2185,3.9108,0.7623,4.0,0.0


In [55]:
summary_lines = []

summary_lines.append("RESULTADO FINAL DO EXPERIMENTO RAG")
summary_lines.append("=" * 80)
summary_lines.append("")
summary_lines.append("Configuração:")
summary_lines.append(json.dumps(asdict(CFG), ensure_ascii=False, indent=2))
summary_lines.append("")
summary_lines.append("=" * 80)
summary_lines.append("MÉTRICAS POR QUESTÃO")
summary_lines.append("=" * 80)
summary_lines.append(df_metricas_questao.round(4).to_string(index=False))
summary_lines.append("")
summary_lines.append("=" * 80)
summary_lines.append("RESUMO POR QUESTÃO (MÉDIA E DESVIO-PADRÃO NAS RODADAS)")
summary_lines.append("=" * 80)
summary_lines.append(df_resumo_questao.round(4).to_string(index=False))
summary_lines.append("")
summary_lines.append("=" * 80)
summary_lines.append("RESUMO FINAL POR MÉTODO")
summary_lines.append("=" * 80)
summary_lines.append(df_resumo_metodo.round(4).to_string(index=False))

summary_txt_path = RESULTS_DIR / "rag_resumo_final.txt"
save_text("\n".join(summary_lines), summary_txt_path)

print(f"Resumo textual salvo em: {summary_txt_path.resolve()}")

Resumo textual salvo em: /home/avelar/outputs_experiment/rag_resumo_final.txt


In [58]:
seed_everything(CFG.base_seed)

print("Recarregando stack do índice...")
resources = load_dense_retriever(CFG)

dense_retriever = resources["dense_retriever"]
parent_docs = load_parent_docs_from_dense_retriever(dense_retriever)
bm25_retriever, tfidf_retriever = build_sparse_retrievers(parent_docs, CFG.retriever_k)

print("Recarregando cross-encoder...")
cross_encoder = CrossEncoder(
    CFG.cross_encoder_model,
    device=CFG.device
)

print("Reconectando ao Ollama...")
ollama_client = build_ollama_client(CFG)
check_ollama_model(ollama_client, CFG.model_name)

print("Runtime pronto para testes.")

Recarregando stack do índice...
Configuração do índice carregada com sucesso.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Chunks-pai carregados do docstore: 12425
Recarregando cross-encoder...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reconectando ao Ollama...
Modelo disponível no Ollama: gemma3:12b
Runtime pronto para testes.


In [59]:
negative_question_item = {
    "documento": "TESTE_NEGATIVO_FORA_DO_ESCOPO",
    "pergunta": "Qual é a capital do Japão?",
    "resposta_esperada": "Não encontrei essa informação no contexto fornecido."
}

for metodo in METHODS:
    print("\n" + "="*100)
    print(f"TESTE NEGATIVO | método = {metodo}")
    print("="*100)

    result = run_single_qa(
        client=ollama_client,
        cfg=CFG,
        method_name=metodo,
        question_item=negative_question_item,
        round_id=0,
        round_seed=999,
        dense_retriever=dense_retriever,
        cross_encoder=cross_encoder,
        bm25_retriever=bm25_retriever,
        tfidf_retriever=tfidf_retriever,
    )

    print("\nPERGUNTA:")
    print(result["pergunta"])

    print("\nRESPOSTA ESPERADA:")
    print(result["resposta_esperada"])

    print("\nRESPOSTA GERADA:")
    print(result["resposta_gerada"])

    print("\nFONTES RECUPERADAS:")
    print(result["fontes_recuperadas"])

    print("\nCONTEXTO RECUPERADO (primeiros 1200 chars):")
    print(result["contexto_recuperado"][:1200])

    print("\nEXACT MATCH:")
    print(calculate_exact_match(result["resposta_gerada"], result["resposta_esperada"]))


TESTE NEGATIVO | método = dense_no_rerank

PERGUNTA:
Qual é a capital do Japão?

RESPOSTA ESPERADA:
Não encontrei essa informação no contexto fornecido.

RESPOSTA GERADA:
O estado do Piauí, situado no noroeste da Região Nordeste do Brasil, tem Teresina como sua capital.

FONTES RECUPERADAS:
["RESOLUÇÃO Nº 54-2024-Aprova a criação do curso de Qualificação Profissional, FIC, em Inglês Intermediário, no IFPI e Anexo.pdf", "RESOLUÇÃO Nº 5-2024-Aprova o Curso de Especialização em  Marketing e Empreendedorismo em Moda,no Instituto Federal de Educação, Ciência e Tecnologia do Piauí (IFPI) e Anexo.pdf", "RESOLUÇÃO Nº 41-2024-Aprova a Criação do curso de Especialização em Atendimento Educacional Especializado no âmbito do Instituto Federal de Educação, Ciência e Tecnologia do Piauí (IFPI e Anexo.pdf", "RESOLUÇÃO Nº 37-2024-Aprova a criação do curso de Especialização em Gestão Escolar, modalidade de Educação a Distância (EaD), no âmbito do IFPI e Anexo.pdf"]

CONTEXTO RECUPERADO (primeiros 1200